# Journey A — Admin สร้างผู้ใช้ใหม่ (ยิง API ทีละขั้น พร้อมอีเมลจริง)

สเปกกำหนดว่าขั้นตอนนี้ **ไม่มีหน้าจอ** — แอดมินเรียก API ตรง ๆ notebook นี้จึงเดินแทนหน้าจอ
โดยยิงทีละ endpoint ให้เห็นทั้ง request และ response ตอนสาธิต

| ขั้น | ใครทำ | endpoint |
| --- | --- | --- |
| 1 | Admin | `POST /api/admin/invitations` — สร้างบัญชี `PENDING` + ออก activation key แล้ว **ส่งอีเมล** |
| 2 | ผู้ถูกเชิญ | `GET /api/auth/invitation?token=` — ตรวจว่าลิงก์ยังใช้ได้ |
| 3 | ผู้ถูกเชิญ | `POST /api/auth/register` — กรอกประวัติและตั้งรหัสผ่าน แล้วระบบ **ส่ง OTP ทางอีเมล** |
| 4 | ผู้ถูกเชิญ | `POST /api/auth/verify-otp` — ยืนยันตัวตน · บัญชีเป็น `ACTIVE` · ผูก role · ปิด key |
| 5 | ผู้ถูกเชิญ | `GET /api/auth/me` — ดูว่าได้ role และหน่วยงานอะไร |

## เปลี่ยนอะไรไปบ้างหลังย้ายสคีมา (2026-08-12)

ถ้าเคยรัน notebook นี้แล้วตอนนี้ขึ้น `400 organizationId` — เพราะสามเรื่องนี้

1. **`Invitation` กลายเป็น `iam.activation_key`** ลำดับใหม่คือสร้าง `user_account` เป็น `PENDING` ก่อน
   แล้วค่อยออก key ให้บัญชีนั้น (ไม่ใช่ผูกคำเชิญไว้กับอีเมลลอย ๆ) · คำตอบจึงคืน
   `activationKeyId` กับ `userAccountId` ไม่ใช่ `invitationId`
2. **role ที่ผูกกับหน่วยงานต้องระบุ `organizationId`** — `ORGANIZATION_USER` และ
   `ORGANIZATION_APPROVER` ส่วน role ฝั่ง BDI ไม่ต้องส่ง ระบบผูกกับหน่วยงาน BDI ให้เอง
3. **รหัส role เปลี่ยนสองตัวและเพิ่มใหม่สองตัว** ของเดิม `BDI_APPROVER` / `BDI_SPECIALIST`
   ใช้ไม่ได้แล้ว

> ⚠️ **หนึ่งหน่วยงานมี `ORGANIZATION_USER` ที่ ACTIVE ได้คนเดียว** (เช่นเดียวกับ
> `ORGANIZATION_APPROVER`) เชิญคนใหม่เข้าหน่วยงานที่มีอยู่แล้ว **คนเดิมจะถูกเพิกถอนสิทธิ์ทันที**
> ที่คนใหม่ยืนยัน OTP เสร็จ · ถ้าไม่อยากทับของเดิม ให้เลือกหน่วยงานอื่นหรือเชิญเป็น role ฝั่ง BDI
> กฎนี้มาจาก sheet `user_account` ใน Excel — ดู `CLAUDE.md` หัวข้อ Auth

## ก่อนเริ่ม

- ต้องมี `requests` (`python3 -m pip install requests`) และเคอร์เนล Python ที่รัน notebook ได้
  เครื่องนี้ยังไม่มี jupyter — เปิดไฟล์นี้ใน VS Code (ต้องมี `ipykernel`) หรือ
  `python3 -m pip install jupyterlab ipykernel` แล้ว `jupyter lab`
- **อีเมลจะถูกส่งจริงก็ต่อเมื่อ checkout นั้นตั้ง `SMTP_USER` / `SMTP_PASS` ไว้แล้ว**
  ตอนนี้มีแค่ `main` ที่ตั้งไว้ — checkout อื่นจะพิมพ์อีเมลลง log แทน (เซลล์ที่ 2 บอกให้ว่าอยู่โหมดไหน)
- ใส่อีเมลของตัวเองในเซลล์ถัดไป ถ้าใช้ Gmail แนะนำ **plus-addressing**
  (`ชื่อคุณ+demo1@gmail.com`) จะได้เชิญซ้ำกี่รอบก็ได้โดยไม่ชนบัญชีเดิม เมลเข้ากล่องเดียวกัน

รายละเอียดของแต่ละหน้าจอที่คู่กับ API เหล่านี้อยู่ใน [`../docs/03-demo-walkthrough.md`](../docs/03-demo-walkthrough.md)

In [7]:
import json
import subprocess
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import requests

# ── แก้ตรงนี้ ────────────────────────────────────────────────────────────
CHECKOUT = Path("/hdd1tb/bdi-project/main")     # checkout ที่จะยิง API ใส่ (main = ส่งอีเมลจริง)
API = "https://bdi-api.thammasorn.org"          # main
                                                # dev_20260812_update-home-page http://localhost:4140

INVITE_EMAIL = "thammasorn.han+demo1@gmail.com"

# ORGANIZATION_USER · ORGANIZATION_APPROVER   ← ใส่ ORGANIZATION_ID หรือเว้นว่าง
# BDI_OFFICER · BDI_DATASET_SPECIALIST · BDI_FINAL_APPROVER · BDI_LEGAL_OFFICER
# SYSTEM_ADMINISTRATOR                        ← ไม่ต้องมี ผูกกับหน่วยงาน BDI ให้เอง
ROLE = "ORGANIZATION_USER"

# เว้นว่าง = เชิญคนที่จะมา "สร้างหน่วยงานของตัวเอง" (จุดเริ่มของ Journey B)
# ระบบจะเตรียมหน่วยงานเปล่ากับคำขอฉบับร่างรอไว้ให้ · ใส่ id = เข้าหน่วยงานที่มีอยู่แล้ว
ORGANIZATION_ID = ""   # เซลล์ถัดไปลิสต์หน่วยงานที่เลือกได้
CID = "1101700200001"  # เลขบัตร 13 หลัก บังคับเฉพาะ role ฝั่งหน่วยงาน (ไม่ตรวจ checksum)
# ─────────────────────────────────────────────────────────────────────────

ORG_SCOPED_ROLES = {"ORGANIZATION_USER", "ORGANIZATION_APPROVER"}

# main ตั้ง APP_URL เป็น https จึงออก session cookie แบบ Secure — ยิงผ่าน
# http://localhost:4000 จะล็อกอินผ่านแต่ทุก call ถัดไปได้ 401 เพราะ cookie ไม่ถูกส่งกลับ
# ใช้ URL https ข้างบนกับ main เสมอ


def env(name: str, default: str = "") -> str:
    """อ่านค่าจาก .env ของ checkout — ไฟล์นี้ไม่อยู่ใน git และเก็บ credential จริง"""
    for line in (CHECKOUT / ".env").read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == name:
            return value.strip().strip('"').strip("'")
    return default


ADMIN_TOKEN = env("ADMIN_API_TOKEN")
APP_URL = env("APP_URL", "http://localhost:3000")
SMTP_USER = env("SMTP_USER")

http = requests.Session()  # เก็บ session cookie ให้อัตโนมัติหลังยืนยัน OTP


def call(method: str, path: str, **kwargs):
    """ยิง API แล้วพิมพ์ทั้งสถานะและ body ให้ดูสด ๆ ตอนสาธิต"""
    response = http.request(method, API + path, timeout=30, **kwargs)
    print(f"{method} {path}  →  {response.status_code} {response.reason}\n")
    try:
        body = response.json()
    except ValueError:
        print(response.text[:800])
        return response, None
    print(json.dumps(body, ensure_ascii=False, indent=2))
    return response, body


def psql(sql: str) -> str:
    """
    ถามฐานข้อมูลของ checkout ตรง ๆ

    ใช้เพราะยังไม่มี endpoint ที่ให้แอดมินลิสต์หน่วยงานได้ด้วย x-admin-token
    (ทุก endpoint ของหน่วยงานต้องมี session) รันได้เฉพาะกับ checkout บนเครื่องนี้
    """
    out = subprocess.run(
        ["docker", "compose", "exec", "-T", "postgres", "psql", "-U", "bdi", "-d", "bdi", "-Atc", sql],
        cwd=CHECKOUT, capture_output=True, text=True,
    )
    return out.stdout.strip() if out.returncode == 0 else f"(psql ล้มเหลว: {out.stderr.strip()[:200]})"


print("พร้อมแล้ว — ยิงไปที่", API)

พร้อมแล้ว — ยิงไปที่ https://bdi-api.thammasorn.org


## 0. ตรวจก่อนว่าระบบพร้อม และอีเมลจะถูกส่งจริงหรือไม่

In [ ]:
call("GET", "/health/ready")

print()
if SMTP_USER:
    print(f"โหมดอีเมล : ส่งจริง ผ่าน {SMTP_USER}")
else:
    print("โหมดอีเมล : พิมพ์ลง log เท่านั้น (SMTP_USER ว่าง)")
    print("            ถ้าต้องการอีเมลจริง ตั้ง SMTP_USER/SMTP_PASS ใน .env แล้ว")
    print("            docker compose up -d backend  —  หรือชี้ CHECKOUT ไปที่ main")

print("ลิงก์ในอีเมลจะชี้ไปที่ :", APP_URL)
print("ADMIN_API_TOKEN       :", "อ่านจาก .env ได้แล้ว" if ADMIN_TOKEN else "ไม่พบใน .env")
print("อีเมลที่จะเชิญ         :", INVITE_EMAIL)
print("role                  :", ROLE)

## 0.5 เลือกหน่วยงาน (เฉพาะ role ฝั่งหน่วยงาน)

`ORGANIZATION_USER` และ `ORGANIZATION_APPROVER` เลือกได้สองแบบ:

- **เว้น `ORGANIZATION_ID` ว่าง** — คนนี้จะเป็นผู้ก่อตั้งหน่วยงานใหม่ ระบบสร้างหน่วยงาน
  เปล่าสถานะ `PENDING_REGISTRATION` พร้อมคำขอฉบับร่างรอไว้ พอเขาล็อกอินแล้วกด
  *สร้างหน่วยงาน* จะเจอร่างใบนั้น นี่คือจุดเริ่มต้นของ Journey B
- **ใส่ id จากรายการข้างล่าง** — เข้าไปเป็นสมาชิกของหน่วยงานที่เปิดใช้งานแล้ว

คอลัมน์ท้ายบอกว่าหน่วยงานนั้น**มีใครถือ role นี้อยู่แล้วหรือยัง** ถ้ามี คนเดิมจะถูกเพิกถอน
เมื่อคนใหม่ยืนยัน OTP เสร็จ — ดีไซน์กำหนดให้หนึ่งหน่วยงานมีผู้ถือ role นี้ได้ทีละคน

> role ฝั่ง BDI ข้ามเซลล์นี้ไปได้เลย

In [3]:
if ROLE in ORG_SCOPED_ROLES:
    rows = psql(
        "select o.id, o.name_th, coalesce(string_agg(u.email, ', '), '—') "
        "from organization.organization o "
        "left join iam.user_role_assignment a on a.organization_id = o.id "
        "  and a.status = 'ACTIVE' "
        "  and a.role_id = (select id from iam.role where code = '" + ROLE + "') "
        "left join iam.user_account u on u.id = a.user_account_id "
        "where o.status = 'ACTIVE' and o.organization_code <> 'BDI' "
        "group by o.id, o.name_th order by o.name_th"
    )
    print(f"หน่วยงานที่ ACTIVE — คอลัมน์ท้ายคือผู้ที่ถือ {ROLE} อยู่ตอนนี้\n")
    for row in rows.splitlines():
        if "|" in row:
            oid, name, holder = row.split("|", 2)
            print(f"  {oid}  {name}")
            print(f"  {'':36}  ปัจจุบัน: {holder}\n")
    print("คัดลอก id ไปใส่ ORGANIZATION_ID ในเซลล์ที่ 1 แล้วรันเซลล์นั้นใหม่")
    print("หรือปล่อยว่างไว้ ถ้าอยากให้คนนี้เป็นผู้ก่อตั้งหน่วยงานใหม่")
else:
    print(f"{ROLE} เป็น role ฝั่ง BDI — ไม่ต้องระบุหน่วยงาน ระบบผูกกับหน่วยงาน BDI ให้เอง")

หน่วยงานที่ ACTIVE — คอลัมน์ท้ายคือผู้ที่ถือ ORGANIZATION_USER อยู่ตอนนี้

  9053091a-fa1a-4c38-b3c9-0b2dcae19a41  สำนักงานสถิติแห่งชาติ
                                        ปัจจุบัน: user@nso.go.th

คัดลอก id ไปใส่ ORGANIZATION_ID ในเซลล์ที่ 1 แล้วรันเซลล์นั้นใหม่
หรือปล่อยว่างไว้ ถ้าอยากให้คนนี้เป็นผู้ก่อตั้งหน่วยงานใหม่


## 1. Admin ส่งคำเชิญ

ป้องกันด้วย `x-admin-token` ไม่ใช่ session เพราะผู้เรียกคือสคริปต์ของผู้ดูแลระบบ ไม่ใช่คนที่ล็อกอิน

ระบบจะ **สร้างบัญชีเป็น `PENDING`** ก่อน แล้วออก activation key ให้บัญชีนั้น
คีย์ถูกเก็บเป็น HMAC-SHA-256 **คำตอบจึงไม่คืนตัวคีย์กลับมา** — มีอยู่ในอีเมลเท่านั้น
เชิญซ้ำอีเมลเดิม คีย์เก่าจะถูก revoke อัตโนมัติ เหลือลิงก์ที่ใช้ได้อันเดียวเสมอ

In [4]:
payload = {"email": INVITE_EMAIL, "role": ROLE}
if ROLE in ORG_SCOPED_ROLES and ORGANIZATION_ID:
    payload["organizationId"] = ORGANIZATION_ID

response, invitation = call(
    "POST",
    "/api/admin/invitations",
    headers={"x-admin-token": ADMIN_TOKEN},
    json=payload,
)

if response.status_code == 201:
    print("\n📧 เปิดกล่องจดหมายของ", INVITE_EMAIL, "— หัวข้อ 'คำเชิญเข้าใช้งาน Government Datahub Platform'")
    print("   บัญชีถูกสร้างเป็น PENDING แล้ว · userAccountId =", invitation["userAccountId"])
    if ROLE in ORG_SCOPED_ROLES and not ORGANIZATION_ID:
        print("   ไม่ได้ระบุหน่วยงาน — ระบบเตรียมหน่วยงานเปล่าไว้ให้แล้ว organizationId =",
              invitation["organizationId"])
        print("   เขาจะได้กรอกชื่อหน่วยงานเองในฟอร์ม Journey B")
elif response.status_code == 409:
    print("\nอีเมลนี้มีบัญชีที่ ACTIVE อยู่แล้ว — เปลี่ยนเป็น +demo2 แล้วรันใหม่")

POST /api/admin/invitations  →  201 Created

{
  "activationKeyId": "60c03bf8-7e38-4a20-9de2-5fc47a1db211",
  "userAccountId": "9c6fe324-f919-40cf-abc5-5e8c3a87a585",
  "email": "thammasorn.h+demo1@gmail.com",
  "role": "ORGANIZATION_USER",
  "roleLabel": "ผู้ดำเนินการของหน่วยงาน",
  "organizationId": "5f4deecb-1d55-4310-a321-a799c2ad6792",
  "expiresAt": "2026-08-20T14:02:41.098Z"
}

📧 เปิดกล่องจดหมายของ thammasorn.h+demo1@gmail.com — หัวข้อ 'คำเชิญเข้าใช้งาน Government Datahub Platform'
   บัญชีถูกสร้างเป็น PENDING แล้ว · userAccountId = 9c6fe324-f919-40cf-abc5-5e8c3a87a585
   ไม่ได้ระบุหน่วยงาน — ระบบเตรียมหน่วยงานเปล่าไว้ให้แล้ว organizationId = 5f4deecb-1d55-4310-a321-a799c2ad6792
   เขาจะได้กรอกชื่อหน่วยงานเองในฟอร์ม Journey B


In [6]:
payload["organizationId"]

KeyError: 'organizationId'

## 2. เปิดอีเมล แล้วเอาลิงก์มาวาง

ในอีเมลมีปุ่ม **เริ่มลงทะเบียน** ชี้ไปที่ `<APP_URL>/register?token=…`
คลิกขวาที่ปุ่ม → คัดลอกลิงก์ แล้ววางในเซลล์ถัดไป (จะวางทั้งลิงก์หรือเฉพาะ token ก็ได้)

> ถ้า checkout อยู่โหมดพิมพ์ลง log ให้ข้ามไปเซลล์ **หาลิงก์จาก log** ท้ายไฟล์นี้

In [ ]:
PASTED = ""  # ← วางลิงก์จากอีเมลตรงนี้

TOKEN = parse_qs(urlparse(PASTED).query).get("token", [PASTED])[0].strip()
print("token :", TOKEN[:12] + "…" if TOKEN else "(ยังไม่ได้วาง)")

call("GET", "/api/auth/invitation", params={"token": TOKEN})

## 3. ผู้ถูกเชิญกรอกข้อมูลและตั้งรหัสผ่าน

อีเมลมาจากคำเชิญ ไม่ได้ส่งขึ้นมา — กันคำเชิญถูกใช้ผิดคน

- เบอร์โทรต้องขึ้นต้น `0` และมี 9–10 หลัก
- รหัสผ่านอย่างน้อย 8 ตัว มีทั้งตัวอักษรและตัวเลข
- **`cid` (เลขบัตร 13 หลัก) บังคับเฉพาะ role ฝั่งหน่วยงาน** — sheet `user_account`
  มาร์กไว้ว่า Required แต่เจ้าหน้าที่ BDI ยังไม่ได้เก็บเลขบัตร จึงบังคับข้างเดียว

ลองกรอกผิดดูได้ — ระบบตอบเป็น `{"error":"validation","fields":{…}}` ระบุทีละช่อง

In [ ]:
PROFILE = {
    "prefix": "นาย",
    "firstName": "ทดสอบ",
    "lastName": "ระบบดี",
    "phone": "0812345678",
    "password": "bdi12345",
}
if ROLE in ORG_SCOPED_ROLES:
    PROFILE["cid"] = CID

response, _ = call("POST", "/api/auth/register", json={"token": TOKEN, **PROFILE})

if response.status_code == 202:
    print("\n📧 กรอกประวัติแล้ว — รหัส OTP 6 หลักกำลังไปที่กล่องจดหมายเดิม")
    print("   บัญชียังเป็น PENDING จนกว่าจะยืนยัน OTP")

## 4. ยืนยันตัวตนด้วย OTP

รหัสอยู่ในหัวข้ออีเมล `รหัสยืนยันตัวตน 123456 — BDI Datahub` มีอายุ 10 นาที
กรอกผิดได้ 5 ครั้ง เกินนั้นต้องขอรหัสใหม่ (`POST /api/auth/resend-otp`)

ขั้นนี้ทำสามอย่างใน transaction เดียวตามลำดับใน sheet `activation_key`:
บัญชีเปลี่ยนเป็น `ACTIVE` · สร้าง `user_role_assignment` · ปิด key เป็น `USED`
แล้วเซิร์ฟเวอร์ตั้ง session cookie ให้ — `http` เก็บ cookie นั้นไว้ต่อ

In [ ]:
OTP = ""  # ← รหัส 6 หลักจากอีเมล

response, _ = call("POST", "/api/auth/verify-otp", json={"token": TOKEN, "code": OTP.strip()})

if response.status_code == 200:
    print("\ncookie ที่ได้รับ :", ", ".join(http.cookies.keys()) or "(ไม่มี)")

## 5. ตรวจผล

บัญชีใหม่เข้าใช้งานได้แล้ว — เข้า `APP_URL` ด้วยอีเมลนี้กับรหัสผ่านที่ตั้งไว้
เพื่อเดินต่อเป็น Journey B (สร้างหน่วยงาน) ได้ทันที

`roles` ใน `/me` อ่านจาก `iam.user_role_assignment` สด ๆ ทุก request ไม่ได้มาจาก cookie

In [ ]:
call("GET", "/api/auth/me")

_, listing = call("GET", "/api/admin/invitations", headers={"x-admin-token": ADMIN_TOKEN})
if listing:
    mine = [i for i in listing["invitations"] if i["userAccount"]["email"] == INVITE_EMAIL]
    print("\nactivation key ของอีเมลนี้ (ล่าสุดอยู่บน) :")
    for item in mine:
        org = item["organization"]["nameTh"] if item.get("organization") else "—"
        print(f"  {item['status']:8} {item['role']['code']:24} {org:40} {item['createdAt']}")
    print("\nUSED = ใช้ไปแล้ว · REVOKED = ถูกแทนที่ด้วยคีย์ใหม่ · EXPIRED = หมดอายุ · ISSUED = ยังใช้ได้")

print("\nเข้าสู่ระบบต่อได้ที่", APP_URL, "ด้วย", INVITE_EMAIL, "/", PROFILE["password"])

---

## หาลิงก์และ OTP จาก log (เมื่อ checkout ไม่ได้ส่งอีเมลจริง)

เมื่อ `SMTP_USER` ว่าง mailer จะพิมพ์เนื้ออีเมล ลิงก์ และ OTP ลง stdout แทนการส่ง

เซลล์นี้แสดงเฉพาะ**รอบล่าสุดของอีเมลที่กำลังเชิญ** — log เก็บทุกรอบไว้ปนกัน
การหยิบลิงก์หรือ OTP ของรอบก่อนมาใช้จะได้ `400` โดยไม่มีอะไรบอกว่าเพราะอะไร


In [ ]:
logs = subprocess.run(
    ["docker", "compose", "logs", "backend", "--tail", "400"],
    cwd=CHECKOUT, capture_output=True, text=True,
).stdout.splitlines()

# ตัดเอาเฉพาะตั้งแต่ครั้งสุดท้ายที่อีเมลนี้ถูกส่งถึง
starts = [i for i, line in enumerate(logs) if INVITE_EMAIL in line]
recent = logs[starts[-1]:] if starts else []

if not recent:
    print(f"ยังไม่พบ {INVITE_EMAIL} ใน log 400 บรรทัดล่าสุด — เชิญไปแล้วหรือยัง?")
else:
    for line in recent:
        if "register?token=" in line or "รหัส OTP" in line or INVITE_EMAIL in line:
            print(line)
    print("\nถ้ามีทั้งลิงก์และ OTP ให้ใช้ของบรรทัดล่างสุดเสมอ — บนสุดคือรอบก่อนหน้า")

## เริ่มรอบใหม่

เปลี่ยน `INVITE_EMAIL` เป็น `+demo2`, `+demo3` … แล้วรันตั้งแต่เซลล์ที่ 1 ใหม่
ถ้าต้องการยกเลิกคีย์ที่ยังใช้ได้โดยไม่ออกใบใหม่ ใช้เซลล์ล่างนี้

In [ ]:
_, listing = call("GET", "/api/admin/invitations", headers={"x-admin-token": ADMIN_TOKEN})
issued = [
    i for i in listing["invitations"]
    if i["userAccount"]["email"] == INVITE_EMAIL and i["status"] == "ISSUED"
]

for item in issued:
    call(
        "POST",
        f"/api/admin/invitations/{item['id']}/revoke",
        headers={"x-admin-token": ADMIN_TOKEN},
        json={"reason": "ยกเลิกจาก notebook"},
    )

if not issued:
    print("ไม่มี activation key ที่ยังใช้ได้ของอีเมลนี้")